# Notebook 09 — Grad-CAM for the metastatic class (GPU, T4 is enough, ~40-60 min)
**Needs:** notebook 03 (`revision/machine_v3`) and notebook 08 (`revision/mets_analysis/mets_predictions.csv`).

The Experiment 1 models were not saved, so the models needed for Grad-CAM are trained again here, with exactly the Experiment 1
settings: task T2 (metastatic vs non-metastatic), EfficientNetV2-S, ImageNet start, seed 42, the same 5 folds and the same inner
validation split inside each machine (10 models). Every patient's map comes from the model of the fold that did **not** see that patient.
As a check, the new out-of-fold probabilities are compared with the saved Experiment 1 (seed 42) probabilities.

Grad-CAM: last convolutional layer of the backbone, one map per view (anterior and posterior), for the metastatic class and, separately,
for the non-metastatic class (for missed patients this shows what drove the wrong answer). Maps are saved for every patient (small numbers, `cams.npz`); figures are made for the patients missed by
all four settings in notebook 08 and for 10 clearly detected patients (5 per machine). Figures show only the patient ID.
The 10 models are saved (`models/`), so later maps need no GPU.

**Runtime:** Runtime > Change runtime type > **T4 GPU**. First `QUICK_TEST = True`, then `False`. Safe to re-run: saved models are reused.

In [ ]:
# settings
import os, sys, subprocess, time, copy, random, json, gc
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=False)
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import timm
from PIL import Image
from sklearn.model_selection import train_test_split

LOCAL_TEST = bool(os.environ.get('BONESCAN_BASE'))
if LOCAL_TEST:
    BASE = os.environ['BONESCAN_BASE']
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
PREP, CV_IN, BSD = BASE + '/prepared', BASE + '/cv5_results', BASE + '/bs80k'
EXP12, METS = BASE + '/revision/machine_v3', BASE + '/revision/mets_analysis'
OUT = BASE + '/revision/gradcam'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'NOT FOUND - select Runtime > Change runtime type > GPU')

QUICK_TEST = True            # first True (short check), then False
MAX_HOURS = 23.0
NAME, INIT, SEED = 'tf_efficientnetv2_s', 'IN', 42
TASK, MAP = 'cancer_vs_noncancer', {'SECONDARY': 1, 'DEGENERATIVE': 0, 'NORMAL': 0}
CAMERAS = ['M1', 'M2']
N_FOLDS, EPOCHS, PATIENCE, BATCH, H, W = 5, 30, 8, 16, 512, 176     # identical to Experiment 1 (notebook 03)
INNER_FRAC = 0.15
RAD_REPO, RAD_FILE = 'Lab-Rasool/RadImageNet', 'InceptionV3.pt'     # not used here (shared code)
PRETRAINED = not LOCAL_TEST
if LOCAL_TEST:
    BATCH = 4
if QUICK_TEST:
    EPOCHS = 1
    OUT = BASE + '/revision/gradcam_quick'
for d in ['models', 'figures', 'logs']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
RUNS = OUT + '/runs.csv'
T_START = time.time()
print('OUT:', OUT, '| QUICK_TEST:', QUICK_TEST)

In [ ]:
# data (prepared_v3 = the final dataset, made by notebook 02)
P3 = BASE + '/prepared_v3'
fix = lambda p: P3 + '/images/' + os.path.basename(str(p))
allp = pd.read_csv(P3 + '/patients.csv')
sp = allp[allp.split == 'pool'][['patient_id', 'machine', 'label', 'ANT', 'POST', 'fold']].reset_index(drop=True)
sp['fold'] = sp.fold.astype(int); sp['split'] = 'pool'      # no separate test set: 5-fold CV inside each machine
assert sp.patient_id.is_unique and sp.label.isin(['SECONDARY', 'DEGENERATIVE', 'NORMAL']).all()
print('classification patients:', len(sp), '| Ind not used:', (allp.split == 'ind').sum(), '| excluded by audit:', (allp.split == 'excluded').sum())
print(pd.crosstab(sp.label, sp.machine, margins=True))

ALL = sp[['patient_id', 'ANT', 'POST', 'machine']].reset_index(drop=True)
IDX = {p: i for i, p in enumerate(ALL.patient_id)}
CAM = dict(zip(ALL.patient_id, ALL.machine))
def load_img(p):
    return np.array(Image.open(fix(p)).convert('L'))
X = torch.from_numpy(np.stack([np.stack([load_img(a), load_img(b)]) for a, b in zip(ALL.ANT, ALL.POST)]))   # N x 2 x H x W uint8
assert X.shape[2:] == (H, W), X.shape
X = X.to(DEVICE)
print('images on', DEVICE, tuple(X.shape), f'{X.numel() / 1e6:.0f} MB')

In [ ]:
# model, GPU augmentation, training
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def augment(img, rng):
    """Same augmentation as the paper (RandomAffine ±7°, ±4 %, 0.93–1.07, white fill; brightness/contrast ±20 %), done on the GPU per image."""
    out = []
    for x in img:                                                   # x: 1 x H x W in [0,1]
        ang = float(rng.uniform(-7, 7)); sc = float(rng.uniform(0.93, 1.07))
        tx, ty = int(round(rng.uniform(-0.04, 0.04) * W)), int(round(rng.uniform(-0.04, 0.04) * H))
        x = TF.affine(x, angle=ang, translate=[tx, ty], scale=sc, shear=[0.0], interpolation=InterpolationMode.NEAREST, fill=[1.0])
        ops = [('b', float(rng.uniform(0.8, 1.2))), ('c', float(rng.uniform(0.8, 1.2)))]
        if rng.random() < 0.5:
            ops = ops[::-1]
        for k, f in ops:
            x = (x * f).clamp(0, 1) if k == 'b' else ((x - x.mean()) * f + x.mean()).clamp(0, 1)
        out.append(x)
    return torch.stack(out)

def to_input(idx, train, rng=None):
    x = X[idx].float() / 255.0                                       # B x 2 x H x W
    a, p = x[:, :1], x[:, 1:]
    if train:
        a, p = augment(a, rng), augment(p, rng)
    norm = lambda t: (t.repeat(1, 3, 1, 1) - MEAN) / STD
    return norm(a), norm(p)

class Pooled(nn.Module):
    """Wraps any feature extractor (e.g. RadImageNet full-model object) so that it returns B x C."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        y = self.m(x)
        if isinstance(y, (tuple, list)):
            y = y[0]
        if hasattr(y, 'logits'):
            y = y.logits
        return F.adaptive_avg_pool2d(y, 1).flatten(1) if y.dim() == 4 else y.flatten(1)

_RAD_CACHE = {}
def radimagenet_backbone():
    if 'm' not in _RAD_CACHE:
        if LOCAL_TEST:
            path = BASE + '/radimagenet_test.pt'
        else:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download(repo_id=RAD_REPO, filename=RAD_FILE)
        obj = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(obj, dict):                                       # file is a state_dict -> rebuild Inception-v3 and load it
            sd = obj.get('state_dict', obj.get('model', obj))
            for pre in ['module.', 'backbone.', 'model.', 'base_model.', 'features.']:
                if all(k.startswith(pre) for k in sd):
                    sd = {k[len(pre):]: v for k, v in sd.items()}
            print('RadImageNet file is a state_dict with', len(sd), 'tensors, e.g.', list(sd)[:3])
            best = None
            for builder in ['timm', 'torchvision']:
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0) if builder == 'timm' else __import__('torchvision').models.inception_v3(weights=None, aux_logits=False, init_weights=False)
                msd = cand.state_dict()
                ok = {k: v for k, v in sd.items() if k in msd and msd[k].shape == v.shape}
                isbb = lambda k: not k.startswith(('fc.', 'AuxLogits'))
                frac = sum(v.numel() for k, v in ok.items() if isbb(k)) / sum(v.numel() for k, v in msd.items() if isbb(k))
                print(f'  {builder} inception_v3: {len(ok)}/{len(msd)} tensors match ({frac:.0%} of backbone weights)')
                if best is None or frac > best[0]:
                    best = (frac, builder, cand, ok)
            frac, builder, m, ok = best
            if frac < 0.95:                                             # saved as nn.Sequential ('0.conv.weight', ...): map by position + shape
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0); msd = cand.state_dict()
                tk = [k for k in msd if not k.startswith(('fc.', 'AuxLogits'))]
                sv = list(sd.values())
                if len(tk) == len(sv) and all(msd[a].shape == v.shape for a, v in zip(tk, sv)):
                    m, ok, frac, builder = cand, dict(zip(tk, sv)), 1.0, 'timm (by position)'
                    print(f'  positional mapping: {len(tk)}/{len(tk)} tensors, all shapes match')
            assert frac > 0.95, ('RadImageNet weights do not match Inception-v3 (only %.0f%% loaded) - check the RadImageNet weight file' % (100 * frac))
            m.load_state_dict(ok, strict=False)
            print(f'  loaded into {builder} inception_v3 ({frac:.0%} of weights)')
        elif isinstance(obj, torch.jit.ScriptModule):
            m = obj
        else:
            m = obj
        for attr in ['fc', 'classifier', 'head']:                       # drop a classification head if present
            if hasattr(m, attr) and isinstance(getattr(m, attr), nn.Module):
                setattr(m, attr, nn.Identity())
        if hasattr(m, 'aux_logits'):
            m.aux_logits = False; m.AuxLogits = None
        _RAD_CACHE['m'] = m
        print('RadImageNet backbone:', type(m).__name__, '| file:', path)
    return Pooled(copy.deepcopy(_RAD_CACHE['m']))

def make_backbone(name, init):
    if init == 'RAD':
        return radimagenet_backbone()
    bb = timm.create_model(name, pretrained=PRETRAINED and init != 'SCRATCH', num_classes=0)
    if init == 'IN':
        return bb
    f = (f'{BSD}/model_bs80k_{name}.pt' if init == 'BS' else
         f'{OUT}/models/model_bs80k_{name}_frac{init[3:]}.pt' if init.startswith('BSf') else       # 'BSf0.10' (independent subsets)
         f'{OUT}/models/model_bs80k_{name}_nested{init[3:]}.pt')                                     # 'BSn0.10' (nested subsets, equal steps)
    sd = torch.load(f, map_location='cpu')
    res = bb.load_state_dict({k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}, strict=False)
    assert not res.missing_keys, f'missing keys in {f}: {res.missing_keys[:3]}'
    return bb

class DualViewNet(nn.Module):
    def __init__(self, name, n, init):
        super().__init__()
        self.backbone = make_backbone(name, init)
        self.backbone.eval()
        with torch.no_grad():
            c = self.backbone(torch.zeros(1, 3, H, W)).shape[1]
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * c, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n))
    def features(self, a, p):
        return torch.cat([self.backbone(a), self.backbone(p)], 1)
    def forward(self, a, p):
        return self.head(self.features(a, p))

def predict(model, idx, amp, feats=False):
    model.eval(); P, Fz = [], []
    with torch.no_grad(), torch.autocast(device_type='cuda', enabled=amp):
        for i in range(0, len(idx), 32):
            a, p = to_input(idx[i:i + 32], False)
            z = model.features(a, p); P.append(F.softmax(model.head(z).float(), 1).cpu())
            if feats:
                Fz.append(z.float().cpu())
    P = torch.cat(P).numpy() if P else np.zeros((0, 1))
    return (P, torch.cat(Fz).numpy().astype(np.float32)) if feats else P

def macro_f1(pred, y, n):
    f = []
    for c in range(n):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def train(name, init, n, tr_idx, tr_y, va_idx, va_y, seed, epochs=EPOCHS, patience=PATIENCE, amp=True):
    set_seed(seed); amp = amp and DEVICE == 'cuda'
    model = DualViewNet(name, n, init).to(DEVICE)
    cnt = np.bincount(tr_y, minlength=n)
    w = torch.tensor(len(tr_y) / (n * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 1e-4}, {'params': model.head.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=amp)
    rng = np.random.default_rng(seed); yt = torch.tensor(tr_y, device=DEVICE)
    best, state, wait, best_ep, ep = -1.0, None, 0, 0, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = rng.permutation(len(tr_idx)); nb = max(len(perm) // BATCH, 1)
        for b in range(nb):
            j = perm[b * BATCH:(b + 1) * BATCH]
            a, p = to_input(tr_idx[j], True, rng)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp):
                loss = loss_fn(model(a, p), yt[j])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        pv = predict(model, va_idx, amp)
        vf = macro_f1(pv.argmax(1), va_y, n) if not np.isnan(pv).any() else -1
        if vf > best:
            best, state, wait, best_ep = vf, copy.deepcopy(model.state_dict()), 0, ep
        else:
            wait += 1
            if wait >= patience:
                break
    if state is not None:
        model.load_state_dict(state)
    return model, dict(best_inner_f1=best, best_epoch=best_ep, epochs_run=ep, amp=amp)

def ids_to_idx(ids):
    return np.array([IDX[p] for p in ids])

def save_preds(path, P, ids, y=None):
    d = pd.DataFrame(P, columns=[f'prob_{i}' for i in range(P.shape[1])])
    if y is not None:
        d.insert(0, 'y_true', y)
    d.insert(0, 'patient_id', list(ids)); d.insert(1, 'camera', [CAM[p] for p in ids]); d.to_csv(path, index=False)

def done_keys():
    return {f[:-5] for f in os.listdir(f'{OUT}/logs') if f.endswith('.json')}

def log_run(row):
    # one small JSON per finished run (safe if Colab disconnects); runs.csv is rebuilt from all of them
    with open(f"{OUT}/logs/{row['key']}.json", 'w') as fh:
        json.dump({k: (v.item() if hasattr(v, 'item') else v) for k, v in row.items()}, fh)
    rows = [json.load(open(f'{OUT}/logs/{f}')) for f in sorted(os.listdir(f'{OUT}/logs')) if f.endswith('.json')]
    pd.DataFrame(rows).to_csv(RUNS, index=False)

def time_left():
    return MAX_HOURS - (time.time() - T_START) / 3600

def run_job(key, exp, name, init, n, seed, tr, va, evals, feat_sets=(), extra=None):
    """tr/va: DataFrames with patient_id and y. evals: {kind: (ids, y or None)}. Saves preds of every eval set."""
    t0 = time.time()
    for amp in [True, False]:
        model, info = train(name, init, n, ids_to_idx(tr.patient_id), tr.y.values, ids_to_idx(va.patient_id), va.y.values, seed, amp=amp)
        outs, bad = {}, False
        for kind, (ids, y) in evals.items():
            res = predict(model, ids_to_idx(ids), info['amp'], feats=kind in feat_sets)
            P = res[0] if kind in feat_sets else res
            bad |= bool(np.isnan(P).any()); outs[kind] = res
        if not bad or not amp or DEVICE != 'cuda':
            break
        print('  NaN -> retrying in full precision'); del model; gc.collect(); torch.cuda.empty_cache()
    for kind, (ids, y) in evals.items():
        res = outs[kind]; P = res[0] if kind in feat_sets else res
        save_preds(f'{OUT}/preds/{key}_{kind}.csv', P, ids, y)
        if kind in feat_sets:
            np.savez_compressed(f'{OUT}/feats/{key}_{kind}.npz', ids=np.array(ids), z=np.clip(res[1], -6e4, 6e4).astype(np.float16))
    row = dict(key=key, exp=exp, model=name, init=init, seed=seed, minutes=round((time.time() - t0) / 60, 2), n_train=len(tr), n_inner=len(va), **info, **(extra or {}))
    for kind, (ids, y) in evals.items():
        if y is not None:
            res = outs[kind]; P = res[0] if kind in feat_sets else res
            row[f'{kind}_f1'] = macro_f1(P.argmax(1), np.asarray(y), n)
    log_run(row)
    print(f"  {key}: inner {info['best_inner_f1']:.3f} | " + ' | '.join(f"{k[:-3]} {v:.3f}" for k, v in row.items() if k.endswith('_f1') and k != 'best_inner_f1') + f" | {row['minutes']} min", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()

def inner_split(df, seed=0):
    for strat in [df.label + '_' + df.machine, df.label, None]:     # class x camera; fall back if a stratum is too small
        try:
            tr, va = train_test_split(df, test_size=INNER_FRAC, stratify=strat, random_state=seed)
            return tr.reset_index(drop=True), va.reset_index(drop=True)
        except ValueError:
            continue

def eta(exp, remaining):
    if not os.path.exists(RUNS):
        return ''
    r = pd.read_csv(RUNS); r = r[r.exp == exp] if 'exp' in r else r.iloc[:0]
    return f'(~{remaining * r.minutes.mean() / 60:.1f} h left for {exp})' if len(r) else ''
print('Ready')

In [ ]:
# train (or load) the 10 fold models and make Grad-CAM maps for every held-out patient
def inner_seed(k, seed):
    return 1000 + k + 100 * (seed - 42)          # identical in all notebooks

def target_layer(bb):
    for nm in ['bn2', 'conv_head']:
        if hasattr(bb, nm):
            return getattr(bb, nm)
    raise RuntimeError('last conv layer not found')

def gradcam(model, i, cls):
    """Grad-CAM for the logit of class `cls` (1 = metastatic, 0 = non-metastatic), one map per view, full precision."""
    model.eval(); acts = []
    def hook(m, inp, out):
        out.retain_grad(); acts.append(out)
    h = target_layer(model.backbone).register_forward_hook(hook)
    try:
        a, p = to_input(np.array([i]), False)
        with torch.enable_grad():
            z = model(a, p); score = z[0, cls]
            model.zero_grad(set_to_none=True); score.backward()
    finally:
        h.remove()
    assert len(acts) == 2, len(acts)                 # anterior, posterior
    maps = []
    for A in acts:
        w = A.grad.mean((2, 3), keepdim=True)
        maps.append(F.relu((w * A).sum(1))[0].detach().float().cpu().numpy())
    return maps[0], maps[1], float(torch.softmax(z.detach().float(), 1)[0, 1])

d = sp.assign(y=sp.label.map(MAP))
REC, CHECK = [], []
for cam in CAMERAS:
    for k in range(N_FOLDS):
        if time_left() <= 0:
            raise SystemExit('MAX_HOURS reached - run all again to resume')
        src = d[d.machine == cam]
        tr_all, outer = src[src.fold != k], src[src.fold == k]
        tr, inner = inner_split(tr_all, seed=inner_seed(k, SEED))
        assert not set(tr.patient_id) & set(outer.patient_id)
        key = f'gc_{TASK}_{NAME}_{INIT}_{cam}_s{SEED}_fold{k}'; mp = f'{OUT}/models/{key}.pt'
        t0 = time.time()
        if os.path.exists(mp):
            model = DualViewNet(NAME, 2, INIT).to(DEVICE); model.load_state_dict(torch.load(mp, map_location=DEVICE)); info = {}
        else:
            for amp in [True, False]:
                model, info = train(NAME, INIT, 2, ids_to_idx(tr.patient_id), tr.y.values, ids_to_idx(inner.patient_id), inner.y.values, SEED, amp=amp)
                if not np.isnan(predict(model, ids_to_idx(outer.patient_id), False)).any() or not amp or DEVICE != 'cuda':
                    break
                print('  NaN -> retrying in full precision')
            torch.save(model.state_dict(), mp)
        for pid, y, lab in zip(outer.patient_id, outer.y, outer.label):
            ma, mpost, prob = gradcam(model, IDX[pid], 1)
            na, npost, _ = gradcam(model, IDX[pid], 0)
            REC.append(dict(patient_id=pid, machine=cam, label=lab, y=int(y), fold=k, prob=prob, ant=ma, post=mpost, ant0=na, post0=npost))
        f = f'{EXP12}/preds/mach_{TASK}_{NAME}_{INIT}_{cam}_s{SEED}_fold{k}_oof.csv'
        if os.path.exists(f):
            old = pd.read_csv(f).set_index('patient_id').prob_1
            new = pd.Series({r['patient_id']: r['prob'] for r in REC if r['machine'] == cam and r['fold'] == k})
            old = old.loc[new.index]
            CHECK.append(dict(machine=cam, fold=k, n=len(new), pearson=float(np.corrcoef(old, new)[0, 1]) if len(new) > 2 else np.nan,
                              same_decision=float(((old > 0.5) == (new > 0.5)).mean())))
        print(f'  {cam} fold {k}: {len(outer)} patients | {"loaded" if not info else "trained, inner F1 %.3f, epoch %d" % (info["best_inner_f1"], info["best_epoch"])} | {(time.time() - t0) / 60:.1f} min', flush=True)
        del model; gc.collect(); torch.cuda.empty_cache()

R = pd.DataFrame([{k: v for k, v in r.items() if k not in ('ant', 'post', 'ant0', 'post0')} for r in REC])
np.savez_compressed(f'{OUT}/cams.npz', ids=R.patient_id.values, machine=R.machine.values, label=R.label.values, prob=R.prob.values,
                    ant=np.stack([r['ant'] for r in REC]).astype(np.float32), post=np.stack([r['post'] for r in REC]).astype(np.float32),
                    ant_nonmet=np.stack([r['ant0'] for r in REC]).astype(np.float32), post_nonmet=np.stack([r['post0'] for r in REC]).astype(np.float32))
R.to_csv(f'{OUT}/probs.csv', index=False)
C = pd.DataFrame(CHECK); C.to_csv(f'{OUT}/check_vs_experiment1.csv', index=False)
print('maps:', len(R), '| map size', REC[0]['ant'].shape, '| metastatic maps that are all zero: %d' % sum(max(r['ant'].max(), r['post'].max()) <= 0 for r in REC))
if len(C):
    print('check vs saved Experiment 1 (seed 42): mean Pearson %.3f, same decision %.1f%%' % (C.pearson.mean(), 100 * C.same_decision.mean()))
yhat = (R.prob > 0.5).astype(int); print('this run, T2 macro-F1: M1 %.3f, M2 %.3f' % tuple(macro_f1(yhat[R.machine == c].values, R.y[R.machine == c].values, 2) for c in CAMERAS))

In [ ]:
# figures: missed by all four settings (notebook 08) and 10 clearly detected patients
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
P20 = pd.read_csv(f'{METS}/mets_predictions.csv')
wide = P20.pivot_table(index='patient_id', columns=['task', 'init'], values='detected')
missed = sorted(wide[(wide == 0).all(1)].index)
t2 = P20[(P20.task == 'T2') & (P20.init == 'IN')]
detected = sum([t2[t2.machine == c].nlargest(5, 'p_metastatic').patient_id.tolist() for c in CAMERAS], [])
print('missed in all four settings:', len(missed), '| detected examples:', len(detected))
REC_BY = {r['patient_id']: r for r in REC}
def up(m):
    t = torch.from_numpy(m)[None, None]
    return F.interpolate(t, size=(H, W), mode='bilinear', align_corners=False)[0, 0].numpy()
def panel(ids, title, fname, ncol=5, cls=1):
    ids = [p for p in ids if p in REC_BY]
    if not ids:
        return
    nrow = int(np.ceil(len(ids) / ncol))
    fig, ax = plt.subplots(nrow, ncol * 2, figsize=(ncol * 2 * 1.25, nrow * 3.6), squeeze=False)
    for a_ in ax.ravel():
        a_.axis('off')
    for j, pid in enumerate(ids):
        r = REC_BY[pid]; i = IDX[pid]; img = X[i].float().cpu().numpy() / 255.0
        ma, mpo = (r['ant'], r['post']) if cls == 1 else (r['ant0'], r['post0'])
        top = max(ma.max(), mpo.max(), 1e-8)
        for v, (nm, m) in enumerate([('ANT', ma), ('POST', mpo)]):
            a_ = ax[j // ncol, (j % ncol) * 2 + v]
            a_.imshow(img[v], cmap='gray', vmin=0, vmax=1); a_.imshow(up(m) / top, cmap='jet', alpha=0.4, vmin=0, vmax=1)
            a_.set_title(f'{pid} {nm}\np={r["prob"]:.2f}' if v == 0 else nm, fontsize=7)
    fig.suptitle(title, fontsize=10); fig.tight_layout(); fig.savefig(f'{OUT}/figures/{fname}', dpi=150); plt.close(fig)
panel(missed, 'Missed metastatic patients: Grad-CAM for the METASTATIC class (T2, EfficientNetV2-S)', 'gradcam_missed.png')
panel(missed, 'Missed metastatic patients: Grad-CAM for the NON-METASTATIC class (what drove the wrong answer)', 'gradcam_missed_nonmet_class.png', cls=0)
panel(detected, 'Clearly detected metastatic patients: Grad-CAM for the METASTATIC class (T2, EfficientNetV2-S)', 'gradcam_detected.png')
for pid in missed + detected:
    if pid in REC_BY:
        panel([pid], pid, f'{"missed" if pid in missed else "detected"}_{pid}.png', ncol=1)
print('figures saved in', OUT + '/figures')
print('DONE')